<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🧑‍⚖️ Human-in-the-Loop（HITL）

**一句话定位**：让 agent 在做有后果的动作之前暂停，等人审核，而不是全自动莽撞执行。

</div>

### 🤔 为什么需要它？

agent 调用的工具可能有**真实世界的副作用**：发邮件、付款、删记录、发布到生产环境。LLM 一旦判断错了就没有撤销键。HITL 在这些高风险步骤前放一个人工检查点，同时让**安全的只读操作自动跑**。

### ⚙️ 在 LangChain 里怎么做

`HumanInTheLoopMiddleware` 通过 `interrupt_on` 标记哪些工具需要审批：

| 工具 | `interrupt_on` 取值 | 行为 |
|---|---|---|
| `read_email` | `False` | 自动运行（只读，安全） |
| `send_email` | `True` | **暂停**，等人决策 |

当一个被标记的工具即将执行时，agent 会**中断**——`invoke()` 不会跑完，而是返回一个 `__interrupt__` payload。执行状态由 `checkpointer`（这里是 `InMemorySaver`）保存，绑定到一个 `thread_id`。人review 后用 `Command(resume=...)` 在**同一个 thread** 上恢复，三种决策可选：

| 决策 | 含义 |
|---|---|
| ✅ **approve** | 按 agent 提议的原样执行工具 |
| ❌ **reject** | 取消这次工具调用，可附带反馈信息给 agent |
| ✏️ **edit** | 修改工具参数后再执行 |

### 🔁 整体流程

```
invoke() → LLM 想调用 send_email
        → middleware 中断  ⏸️
        → 返回 __interrupt__（agent 暂停，状态已 checkpoint）
        → 人review 这次提议的动作
        → invoke(Command(resume={...}), 同一个 thread_id) ▶️
        → approve / reject / edit → agent 继续跑
```

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**：`thread_id` 是恢复的锚点——resume 时必须用和中断时**完全相同**的 `thread_id`，否则 checkpointer 找不到被暂停的那次执行，等于从零重新开始。

</div>

下面依次演示 approve / reject / edit 三种决策。

In [10]:
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
from langchain.tools import tool, ToolRuntime


@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]


@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [12]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [13]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Please read my email and send a response immediately. Send the reply now in the same thread."
            )
        ],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.",
    },
    config=config,
)

In [14]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads-up. '
                                                                          'We '
                                                                          'can '
                                                                          'reschedule. '
                                                                          'Are '
     

In [15]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John,\n\nNo problem—thanks for the heads-up. We can reschedule. Are you available tomorrow at 10:00 AM or 2:00 PM? If neither works, please suggest a time that suits you, and I’ll adjust. If you’d prefer, we can keep it quick and do a 15-minute catch-up at a time that fits.\n\nBest regards,\nSeán'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John,\\n\\nNo problem—thanks for the heads-up. We can reschedule. Are you available tomorrow at 10:00 AM or 2:00 PM? If neither works, please suggest a time that suits you, and I’ll adjust. If you’d prefer, we can keep it quick and do a 15-minute catch-up at a time that fits.\\n\\nBest regards,\\nSeán'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='be5a2bfc6c2f21763afc50d43f6cadf2')]


In [16]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for the heads-up. We can reschedule. Are you available tomorrow at 10:00 AM or 2:00 PM? If neither works, please suggest a time that suits you, and I’ll adjust. If you’d prefer, we can keep it quick and do a 15-minute catch-up at a time that fits.

Best regards,
Seán


## ✅ Approve（批准）

In [ ]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

## ❌ Reject（拒绝）

In [17]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads-up. '
                                                                          'We '
                                                                          'can '
                                                                          'reschedule. '
                                                                          'Are '
     

In [18]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for the heads-up. We can reschedule. Are you available tomorrow at 10:00 AM or 2:00 PM? If neither works, please suggest a time that suits you, and I’ll adjust. If you’d prefer, we can keep it quick and do a 15-minute catch-up at a time that fits.

Your merciful leader, Seán


## ✏️ Edit（修改）

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：HITL 不是"要不要自动化"的开关，而是把高风险工具和安全工具分开治理——`interrupt_on` 精确控制谁需要人、谁不需要，`Command(resume=...)` 则是人把决策喂回暂停中的 agent 的唯一入口。

</div>

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)